# Notebook 1: Exploratory Data Analysis — CAFA6

This notebook explores the CAFA6 dataset:
- Sequence length distribution
- GO term frequency distribution per ontology
- Label sparsity and co-occurrence
- Train/Val/Test split statistics

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from preprocessing import parse_fasta, parse_go_annotations, filter_sequences

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

Download CAFA6 data from Kaggle: https://www.kaggle.com/competitions/cafa-6-protein-function-prediction

In [ ]:
FASTA_PATH = '../data/raw/train_sequences.fasta'
ANNOT_PATH = '../data/raw/train_annotations.tsv'

sequences = parse_fasta(FASTA_PATH)
sequences = filter_sequences(sequences)
print(f'Total sequences after filtering: {len(sequences):,}')

In [ ]:
annotations = parse_go_annotations(ANNOT_PATH)
print(annotations.head())
print(f'\nAnnotation counts per ontology:')
print(annotations['ontology'].value_counts())

## 2. Sequence Length Distribution

In [ ]:
lengths = [len(s) for s in sequences.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(lengths, bins=100, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Sequence Length (aa)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sequence Length Distribution')
axes[0].axvline(np.median(lengths), color='red', linestyle='--', label=f'Median={np.median(lengths):.0f}')
axes[0].legend()

axes[1].hist(np.log10(lengths), bins=80, color='coral', edgecolor='white')
axes[1].set_xlabel('log10(Sequence Length)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log-Scale Sequence Length Distribution')

plt.tight_layout()
plt.savefig('../reports/seq_length_dist.png', dpi=150)
plt.show()

print(f'Min: {min(lengths)} | Max: {max(lengths)} | Median: {np.median(lengths):.0f} | Mean: {np.mean(lengths):.0f}')
for pct in [50, 75, 90, 95, 99]:
    print(f'  {pct}th percentile: {np.percentile(lengths, pct):.0f} aa')

## 3. GO Term Frequency Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, ont in zip(axes, ['BPO', 'MFO', 'CCO']):
    sub = annotations[annotations['ontology'] == ont]
    term_counts = sub.groupby('go_term')['protein_id'].nunique().sort_values(ascending=False)
    
    ax.hist(term_counts.values, bins=60, log=True, color='mediumseagreen', edgecolor='white')
    ax.set_xlabel('Number of Annotated Proteins per GO Term')
    ax.set_ylabel('Count (log)')
    ax.set_title(f'{ont} — {len(term_counts):,} terms')
    
    for cutoff in [10, 50, 100]:
        n = (term_counts >= cutoff).sum()
        print(f'{ont}: {n:,} terms with ≥{cutoff} proteins')

plt.tight_layout()
plt.savefig('../reports/go_term_frequency.png', dpi=150)
plt.show()

## 4. Label Statistics

In [ ]:
protein_ids = list(sequences.keys())
annotated_pids = set(annotations['protein_id'].unique())
overlap = len(set(protein_ids) & annotated_pids)
print(f'Proteins with sequence AND annotation: {overlap:,} / {len(protein_ids):,}')

# Terms per protein
terms_per_protein = annotations[annotations['protein_id'].isin(protein_ids)].groupby('protein_id')['go_term'].nunique()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(terms_per_protein.values, bins=80, color='mediumpurple', edgecolor='white')
ax.set_xlabel('Number of GO Terms per Protein')
ax.set_ylabel('Count')
ax.set_title('GO Term Cardinality per Protein (all ontologies)')
ax.axvline(terms_per_protein.median(), color='red', linestyle='--', label=f'Median={terms_per_protein.median():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/terms_per_protein.png', dpi=150)
plt.show()

## 5. Top 30 Most Frequent GO Terms

In [ ]:
for ont in ['BPO', 'MFO', 'CCO']:
    sub = annotations[annotations['ontology'] == ont]
    top30 = sub.groupby('go_term')['protein_id'].nunique().nlargest(30)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(top30.index[::-1], top30.values[::-1], color='steelblue')
    ax.set_xlabel('Number of Proteins')
    ax.set_title(f'Top 30 GO Terms — {ont}')
    plt.tight_layout()
    plt.savefig(f'../reports/top30_{ont.lower()}.png', dpi=150)
    plt.show()

## 6. Summary

In [ ]:
print('=== CAFA6 Dataset Summary ===')
print(f'Total filtered sequences : {len(sequences):,}')
print(f'Proteins with annotations: {overlap:,}')
print(f'Total GO annotations     : {len(annotations):,}')
print(f'Unique GO terms          : {annotations["go_term"].nunique():,}')
print()
for ont in ['BPO', 'MFO', 'CCO']:
    sub = annotations[annotations['ontology'] == ont]
    print(f'{ont}: {sub["go_term"].nunique():,} terms, {sub["protein_id"].nunique():,} proteins')